**Select chat model**

In [5]:
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv()
# assign key from env to langchain/openai
os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")

from langchain.chat_models import init_chat_model
model = init_chat_model("mistral-small", model_provider="mistralai", temperature=0.2)

**Track tokens used**

In [6]:
from langchain_mistralai import ChatMistralAI
llm = ChatMistralAI(model="mistral-small")

# response = model.invoke("Say hello in German")
# response.usage_metadata
# print(response)

**invoke structured prompt**
- generate n idioms in different german level

In [ ]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative german Teacher. You generate german idioms accordings to given topic for given level"),
    ("human", "Provide {number_idioms} idioms with {topic} for level A2"),
])
messages = prompt.format_messages(number_idioms=5, topic="food")

response = model.invoke(messages)
print(response.content)

### Generate idioms with given user's input, and put up a question

In [7]:
# Using PromptTemplate to set up prompt
from langchain.prompts import PromptTemplate
# call chain functions
from langchain.chains import LLMChain
# define model -> is done above

# define and chain idiom template
idiom_prompt = PromptTemplate(
    input_variables=["nbr_idioms", "topic", "level", "answer"],
    template =(
    "You are a creative German teacher. You generate German idioms according to a given topic for a given level.\n\n"
    "Provide {nbr_idioms} idioms about {topic} for level {level}."
    "After providing reponse ask user to make an example with one of given idiom"
    )
)
idiom_chain = LLMChain(llm=llm, prompt=idiom_prompt)

# fetch user's input
nbr_idioms = input("Enter number: ")
topic = input("Enter topic: ")
level = input("Enter level: ")

# collect user input
user_input = {
    "nbr_idioms": 3,
    "topic": "animal",
    "level": "A1",
}

# run and print the chain
idiom_response = idiom_chain.run(user_input)
print("\nGenerated idioms: \n", idiom_response)



Generated idioms: 
 Sure, I'd be happy to help! Here are three German idioms about animals that are suitable for level A1:

1. Das ist ein elefantenhautes Problem - This is a huge problem
(Literally: That is an elephant-sized problem)
2. Wie die Lemminge - Like lemmings
(Literally: Like the lemmings)
This idiom is used to describe a situation where a large group of people is following each other blindly, without considering the consequences.
3. Jemanden an die Löffel nehmen - To take someone by the spoon
(Literally: To take someone by the spoon)
This idiom is used to describe a situation where someone is trying to manipulate or control another person.

Now, could you please make an example with one of these idioms? For instance, you could use the first idiom to describe a difficult situation, like this:

Die Reparaturkosten für mein Auto sind ein elefantenhautes Problem. (The repair costs for my car are a huge problem.)


### generate evaluation based on the answer given by user

In [8]:
# define and chain evaluation_prompt
evaluation_prompt = PromptTemplate(
    input_variable = ["user_example", "idiom"],
    template=(
        "You are a German language teacher. Evaluate following sentece:\n"
        "\"{user_example}\"\n"
        "Did the user correctly use the idiom \"{idiom}\"? Provide detailed feedback in simple language" 
    )
) 
evaluation_chain = LLMChain(llm=llm, prompt = evaluation_prompt)

#  Ask user to pick an idiom and write a sentence
idiom_chosen = input("\nChoose one idiom from above to use: ")
user_example = input(f"Write a sentence using the idiom '{idiom_chosen}': ")

# Evaluate the sentence
evaluation_response = evaluation_chain.run({
    "user_example": user_example,
    "idiom": idiom_chosen
})
print("\nEvaluation:\n", evaluation_response)



Evaluation:
 The sentence "Die Geriegung hat immer wieder versucht Bürgern an die Löffel zu nehmen" is not correct. The idiom "jemanden an die Löffel nehmen" is used in German to say that someone or something is trying to cheat or deceive someone else. However, the sentence is not grammatically correct and makes little sense as it is written. The word "Geriegung" does not exist in the German language. If the user wanted to say that the government (Regierung) kept trying to deceive citizens, a possible correct sentence could be: "Die Regierung hat immer wieder versucht, Bürger an der Nase herumzuführen".
